In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load seluruh dataset
DATA_DIR = Path("/content")

files = {
    "categories": "categories.csv",
    "suppliers": "suppliers.csv",
    "products": "products.csv",
    "warehouses": "warehouses.csv",
    "sales_orders": "sales_orders.csv",
    "sales_order_items": "sales_order_items.csv",
    "purchase_orders": "purchase_orders.csv",
    "purchase_order_items": "purchase_order_items.csv",
    "inventory_snapshots": "inventory_snapshots.csv",
}

df = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in files.items()
}

# 2. Ringkasan ukuran data
print("=== UKURAN DATA ===")
for name, data in df.items():
    print(f"{name}: {data.shape}")

# 3. Cek missing values
print("\n=== MISSING VALUES ===")
for name, data in df.items():
    missing = data.isna().sum().sum()
    if missing > 0:
        print(f"{name}: {missing} nilai kosong")

# 4. Cek primary key duplikat
print("\n=== DUPLIKAT PRIMARY KEY ===")
primary_keys = {
    "categories": "category_id",
    "suppliers": "supplier_id",
    "products": "product_id",
    "warehouses": "warehouse_id",
    "sales_orders": "sales_order_id",
    "sales_order_items": "sales_item_id",
    "purchase_orders": "purchase_order_id",
    "purchase_order_items": "purchase_item_id",
    "inventory_snapshots": "snapshot_id",
}

for name, key in primary_keys.items():
    if key in df[name].columns:
        duplicates = df[name][key].duplicated().sum()
        print(f"{name}: {duplicates}")

# 5. Analisis tren penjualan bulanan
orders = df["sales_orders"].copy()
items = df["sales_order_items"].copy()

orders["order_date"] = pd.to_datetime(orders["order_date"])

completed = orders.loc[
    orders["order_status"] == "Completed"
].merge(
    items,
    on="sales_order_id",
    how="inner",
    validate="one_to_many"
)

completed["revenue"] = (
    completed["quantity"] * completed["unit_selling_price"]
)
completed["profit"] = (
    completed["quantity"]
    * (
        completed["unit_selling_price"]
        - completed["unit_cost_at_sale"]
    )
)

completed["bulan"] = completed["order_date"].dt.to_period("M")

monthly = completed.groupby("bulan").agg(
    total_order=("sales_order_id", "nunique"),
    total_unit=("quantity", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum")
).reset_index()

print("\n=== TREN PENJUALAN BULANAN ===")
display(monthly.round(2))

# 6. Analisis stok terbaru
inventory = df["inventory_snapshots"].copy()
products = df["products"].copy()

inventory["snapshot_month"] = pd.to_datetime(
    inventory["snapshot_month"]
)

latest_month = inventory["snapshot_month"].max()

latest_inventory = inventory.loc[
    inventory["snapshot_month"] == latest_month
].merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    suffixes=("_inventory", "_product")
)

# Menghitung nilai persediaan berdasarkan unit_cost dari inventory_snapshots
latest_inventory["inventory_value"] = (
    latest_inventory["ending_stock"]
    * latest_inventory["unit_cost_inventory"]
)

# Menentukan produk yang perlu restock
latest_inventory["perlu_restock"] = (
    latest_inventory["ending_stock"]
    <= latest_inventory["reorder_point"]
)

print("\n=== RINGKASAN INVENTORY TERBARU ===")
print("Bulan snapshot:", latest_month.date())

print(
    "Total nilai persediaan:",
    round(latest_inventory["inventory_value"].sum(), 2)
)

print(
    "Kombinasi produk-gudang perlu restock:",
    int(latest_inventory["perlu_restock"].sum())
)

# 7. Produk dengan unit terjual tertinggi
product_demand = completed.groupby("product_id").agg(
    total_unit_terjual=("quantity", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum")
).reset_index()

product_demand = product_demand.merge(
    products[["product_id", "product_name"]],
    on="product_id",
    how="left",
    validate="one_to_one"
).sort_values("total_unit_terjual", ascending=False)

print("\n=== TOP 10 PRODUK BERDASARKAN UNIT TERJUAL ===")
display(product_demand.head(10).round(2))


=== UKURAN DATA ===
categories: (20, 2)
suppliers: (300, 5)
products: (2000, 9)
warehouses: (10, 5)
sales_orders: (60000, 4)
sales_order_items: (180076, 6)
purchase_orders: (8000, 7)
purchase_order_items: (24086, 6)
inventory_snapshots: (240000, 10)

=== MISSING VALUES ===
purchase_orders: 938 nilai kosong

=== DUPLIKAT PRIMARY KEY ===
categories: 0
suppliers: 0
products: 0
warehouses: 0
sales_orders: 0
sales_order_items: 0
purchase_orders: 0
purchase_order_items: 0
inventory_snapshots: 0

=== TREN PENJUALAN BULANAN ===


,bulan,total_order,total_unit,revenue,profit
0,2025-01,4668,77964,3.832913e+10,1.234533e+10
1,2025-02,4244,69945,3.455621e+10,1.116797e+10
2,2025-03,4693,77991,3.883009e+10,1.257675e+10
3,2025-04,4492,74586,3.588955e+10,1.155008e+10
4,2025-05,4680,77232,3.788404e+10,1.215204e+10
5,2025-06,4511,73793,3.700370e+10,1.187878e+10
6,2025-07,4646,76279,3.768783e+10,1.207312e+10
7,2025-08,4680,76617,3.839221e+10,1.247777e+10
8,2025-09,4631,75513,3.755966e+10,1.202791e+10
9,2025-10,4716,77405,3.759609e+10,1.217699e+10



=== RINGKASAN INVENTORY TERBARU ===
Bulan snapshot: 2025-12-01
Total nilai persediaan: 1015886269581.64
Kombinasi produk-gudang perlu restock: 4455

=== TOP 10 PRODUK BERDASARKAN UNIT TERJUAL ===


,product_id,total_unit_terjual,revenue,profit,product_name
1580,1581,636,1.499872e+09,5.896432e+08,Product 1581
1167,1168,635,8.408557e+08,3.100629e+08,Product 1168
773,774,632,2.291772e+08,9.459391e+07,Product 0774
240,241,623,9.452592e+06,4.162219e+06,Product 0241
1810,1811,621,3.858065e+08,1.658653e+08,Product 1811
1638,1639,613,6.160835e+07,1.900510e+07,Product 1639
1042,1043,611,3.414882e+08,8.439175e+07,Product 1043
1817,1818,611,6.902395e+07,1.387130e+07,Product 1818
1382,1383,611,7.117496e+06,1.611488e+06,Product 1383
1938,1939,607,1.996546e+08,7.865654e+07,Product 1939


In [ ]:

# 8. Analisis risiko kehabisan stok
# Menggunakan penjualan Completed sepanjang 2025

# Total unit terjual per produk
demand = (
    completed.groupby("product_id", as_index=False)
    .agg(total_unit_terjual=("quantity", "sum"))
)

# Rata-rata permintaan harian selama 365 hari
demand["rata_rata_harian"] = demand["total_unit_terjual"] / 365

# Ambil stok terbaru dan gabungkan permintaan
stock_risk = latest_inventory.merge(
    demand,
    on="product_id",
    how="left",
    validate="many_to_one"
)

stock_risk["total_unit_terjual"] = (
    stock_risk["total_unit_terjual"].fillna(0)
)
stock_risk["rata_rata_harian"] = (
    stock_risk["rata_rata_harian"].fillna(0)
)

# Estimasi berapa hari stok bertahan
stock_risk["estimasi_hari_stok"] = np.where(
    stock_risk["rata_rata_harian"] > 0,
    stock_risk["ending_stock"] / stock_risk["rata_rata_harian"],
    np.nan
)

# Prioritaskan stok <= reorder point dengan permintaan tertinggi
priority = stock_risk.loc[
    stock_risk["perlu_restock"]
].sort_values(
    ["estimasi_hari_stok", "total_unit_terjual"],
    ascending=[True, False]
)

print("=== 20 PRIORITAS RESTOCK ===")
display(
    priority[
        [
            "product_id",
            "product_name",
            "ending_stock",
            "reorder_point",
            "total_unit_terjual",
            "rata_rata_harian",
            "estimasi_hari_stok",
        ]
    ].head(20).round(2)
)


=== 20 PRIORITAS RESTOCK ===


,product_id,product_name,ending_stock,reorder_point,total_unit_terjual,rata_rata_harian,estimasi_hari_stok
5915,592,Product 0592,20,97,601,1.65,12.15
18104,1811,Product 1811,21,36,621,1.70,12.34
10061,1007,Product 1007,20,48,589,1.61,12.39
18170,1818,Product 1818,21,31,611,1.67,12.55
2788,279,Product 0279,20,55,576,1.58,12.67
7726,773,Product 0773,20,65,568,1.56,12.85
2409,241,Product 0241,22,62,623,1.71,12.89
3308,331,Product 0331,21,85,581,1.59,13.19
5744,575,Product 0575,21,85,581,1.59,13.19
4626,463,Product 0463,20,23,553,1.52,13.20
